# SIEM exploration — synthetic OCSF dataset

This notebook does four kinds of analysis against the synthetic OCSF
dataset generated by `scripts/generate_synthetic_data.py`:

1. **Capacity / volume** — events per class / day / hour; cardinality.
2. **Threat hunting** — failed-auth patterns, top bad IPs, cross-source
   pivots, find the 5 injected incident scenarios.
3. **Detection-rule prototyping** — Sigma-style rules in pandas:
   brute force, off-hours admin, Vault permission spikes, multi-region
   API activity.
4. **Compliance evidence** — severity distribution, MITRE coverage,
   audit completeness.

## Regenerating the dataset

```bash
# Default: 30 days, ~5 GB Parquet (scale=8). Adjust --scale up or down.
python3 scripts/generate_synthetic_data.py --scale 8 --out data/synthetic/ocsf

# Quick smoke test (~50 MB, finishes in seconds):
python3 scripts/generate_synthetic_data.py --scale 0.05 --out data/synthetic/ocsf
```

Output layout: `data/synthetic/ocsf/class_uid=NNNN/event_day=YYYY-MM-DD/part-*.parquet`
— Hive-partitioned, auto-discoverable by `pyarrow.dataset`.

## Injected anomalies (so the notebook has signal to find)

| Scenario | When | What |
|---|---|---|
| Brute-force burst | Day 14, 12:00 UTC | 200 failed sshd attempts on `carol@corp.example.com` from `185.220.101.42`, plus 1 successful login |
| GuardDuty critical chain | Day 20, 09:00 UTC | severity 5 finding → PagerDuty triggered → ack 10m → resolved 2h |
| Vault permission-denied spike | Day 10, 14:00 UTC | 50 denied secret reads by `rogue-svc-account` from `45.155.205.7` |
| Lateral movement | Day 5, 23:00 UTC | One principal hitting AWS APIs across 5 regions within 10 minutes |
| MFA fraud cluster | Day 25, 03:00 UTC | 8 Duo `fraud` results across 5 users — exec-targeted, from `185.220.101.x` |


## Setup

In [ ]:
# Standard analytical stack — pyarrow.dataset for the Hive-partitioned read,
# pandas for the filter/group/aggregate dance, matplotlib for the charts.
import pyarrow.dataset as ds
import pyarrow as pa
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timezone
from pathlib import Path

# Repo-rooted dataset path. Override DATA_ROOT if you generated elsewhere.
DATA_ROOT = Path("../data/synthetic/ocsf")
if not DATA_ROOT.is_dir():
    DATA_ROOT = Path("data/synthetic/ocsf")  # if running from repo root
assert DATA_ROOT.is_dir(), f"No dataset at {DATA_ROOT}. Run scripts/generate_synthetic_data.py first."

# Explicit partition typing — pyarrow's auto-inference picks int32 for
# small class_uids and int64 for larger ones (like 201001 for the Windows
# extension), which then can't merge cross-partition. Fix the types
# up-front and the dataset loads cleanly.
DS = ds.dataset(
    DATA_ROOT, format="parquet",
    partitioning=ds.partitioning(
        pa.schema([("class_uid", pa.int64()), ("event_day", pa.string())]),
        flavor="hive",
    ),
)
print(f"Dataset: {DATA_ROOT}")
print(f"Schema (top): {[f.name for f in DS.schema][:20]}...")
print(f"Partition keys: {[f.name for f in DS.schema if f.name in ('class_uid', 'event_day')]}")


In [ ]:
# Dataset stats — total rows, partition count, time range, disk size.
total_rows = DS.count_rows()
part_files = list(DATA_ROOT.rglob("*.parquet"))
total_bytes = sum(f.stat().st_size for f in part_files)

# Class/day coverage from the partition paths (cheaper than scanning all rows).
class_uids = sorted({p.parent.parent.name.split('=')[1] for p in part_files})
days       = sorted({p.parent.name.split('=')[1]        for p in part_files})

print(f"Total events:    {total_rows:,}")
print(f"Part files:      {len(part_files):,}")
print(f"On-disk size:    {total_bytes / 1024 / 1024 / 1024:.2f} GB ({total_bytes / total_rows:.0f} bytes/event)")
print(f"OCSF classes:    {len(class_uids)} ({', '.join(class_uids[:8])}…)")
print(f"Date range:      {days[0]}  →  {days[-1]}  ({len(days)} days)")


## §1 — Capacity & volume

Sizing questions: which classes dominate ingest, what's the per-day
shape, where are the peak hours, how many distinct entities (users
/ IPs / accounts) are we tracking?

In [ ]:
# Events per OCSF class — what dominates ingest?
# Use the partition column directly; no row scan needed for the count.
by_class = (
    DS.to_table(columns=["class_uid"])
      .to_pandas()
      .groupby("class_uid").size()
      .sort_values(ascending=False)
)
print(by_class.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
by_class.plot(kind="bar", ax=ax, color="#0969da")
ax.set_xlabel("OCSF class_uid")
ax.set_ylabel("event count")
ax.set_title("Events per OCSF class (log scale)")
ax.set_yscale("log")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Events per day — time-series, all classes stacked OR per-class lines.
by_day_class = (
    DS.to_table(columns=["event_day", "class_uid"])
      .to_pandas()
      .groupby(["event_day", "class_uid"]).size()
      .unstack(fill_value=0)
      .sort_index()
)

# Drop sparse classes (<10K total events across the window) to keep the
# chart readable.
big_classes = by_day_class.sum().nlargest(8).index
focus = by_day_class[big_classes]

fig, ax = plt.subplots(figsize=(11, 5))
focus.plot(ax=ax, linewidth=1.5)
ax.set_xlabel("event_day")
ax.set_ylabel("events / day")
ax.set_title("Top-8 OCSF classes — ingest volume over 30 days")
ax.legend(title="class_uid", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Peak-hour heatmap (day of week × hour of day) — when does ingest spike?
# Sample 5% so the to_pandas() fits in memory; volume patterns are stable.
sample = DS.to_table(columns=["time"]).to_pandas().sample(frac=0.05, random_state=42)
sample["dt"] = pd.to_datetime(sample["time"], unit="ms", utc=True)
sample["dow"] = sample["dt"].dt.day_name()
sample["hour"] = sample["dt"].dt.hour

heat = sample.groupby(["dow", "hour"]).size().unstack(fill_value=0)
# Reorder rows Monday→Sunday for readability.
heat = heat.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(24)); ax.set_xticklabels(range(24))
ax.set_yticks(range(7)); ax.set_yticklabels(heat.index)
ax.set_xlabel("hour of day (UTC)")
ax.set_title("Ingest density — day of week × hour (5% sample)")
plt.colorbar(im, ax=ax, label="events (sampled)")
plt.tight_layout()
plt.show()


In [ ]:
# Cardinality — how many distinct users / IPs / accounts are we tracking?
# Pull only the fields we need; pyarrow filters at scan time.
ent = DS.to_table(columns=["actor", "user", "src_endpoint", "cloud"]).to_pandas()

def deep_get(s, key):
    return s.apply(lambda d: (d or {}).get(key) if isinstance(d, dict) else None)

ent["actor_user"]   = deep_get(ent["actor"],        "user").apply(lambda u: (u or {}).get("name") if isinstance(u, dict) else None)
ent["user_name"]    = deep_get(ent["user"],         "name")
ent["src_ip"]       = deep_get(ent["src_endpoint"], "ip")
ent["cloud_acct"]   = deep_get(ent["cloud"],        "account").apply(lambda a: (a or {}).get("uid") if isinstance(a, dict) else None)

stats = pd.Series({
    "Distinct actor.user.name": ent["actor_user"].dropna().nunique(),
    "Distinct user.name":       ent["user_name"].dropna().nunique(),
    "Distinct src_endpoint.ip": ent["src_ip"].dropna().nunique(),
    "Distinct cloud.account":   ent["cloud_acct"].dropna().nunique(),
})
print(stats.to_string())


## §2 — Threat hunting

Now we hunt for the 5 injected scenarios. Each subsection below
demonstrates a finding the notebook can surface without needing a
real SIEM.

In [ ]:
# Failed-authentication distribution — class_uid 3002 (Authentication)
# with status_id != 1 (Success). Includes Okta DENY, Duo fraud/denied,
# sshd "Failed password", windows_event_log 4625.
auth = DS.to_table(
    filter=(ds.field("class_uid") == 3002),
    columns=["time", "actor", "user", "src_endpoint", "status_id", "status", "status_detail", "metadata"],
).to_pandas()
auth["dt"]        = pd.to_datetime(auth["time"], unit="ms", utc=True)
auth["actor"]     = auth["actor"].apply(lambda d: (d or {}).get("user", {}).get("name") if isinstance(d, dict) else None)
auth["src_ip"]    = auth["src_endpoint"].apply(lambda d: (d or {}).get("ip") if isinstance(d, dict) else None)
auth["product"]   = auth["metadata"].apply(lambda d: ((d or {}).get("product") or {}).get("name") if isinstance(d, dict) else None)

failed = auth[auth["status_id"] != 1]
print(f"Auth events:     {len(auth):,}  ({len(failed):,} failures, "
      f"{len(failed)/max(1,len(auth))*100:.1f}%)")
print()
print("Failures by product:")
print(failed["product"].value_counts().head(10).to_string())
print()
print("Failures by status_detail (top 10):")
print(failed["status_detail"].fillna("(none)").value_counts().head(10).to_string())


In [ ]:
# Top source IPs by failed-auth count — typical SOC starting point.
top_bad_ips = failed.groupby("src_ip").size().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(11, 5))
top_bad_ips.plot(kind="barh", ax=ax, color="#cf222e")
ax.set_xlabel("failed auth events")
ax.set_title("Top 15 source IPs by failed authentication count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# IP-reputation buckets — split top IPs by their synthetic "persona".
# The generator drew from these ranges, so we can re-bucket and see
# whether failures concentrate in the known-bad range (they should).
def ip_bucket(ip):
    if ip is None: return "unknown"
    if ip.startswith("10.0."):           return "corp internal"
    if ip.startswith("192.0.2."):        return "corp VPN"
    if ip.startswith("203.0.113."):      return "cloud egress"
    if ip.startswith("198.51.100."):     return "partner"
    if ip.startswith(("185.220.101.","45.155.205.","194.32.122.")):
        return "known-bad"
    return "other"

failed["ip_bucket"] = failed["src_ip"].apply(ip_bucket)
bucket_failures = failed["ip_bucket"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bucket_failures.plot(kind="bar", ax=ax, color=["#1f883d","#0969da","#bf8700","#57606a","#cf222e","#888"])
ax.set_ylabel("failed auth events")
ax.set_title("Failed authentications by source-IP reputation bucket")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()
print()
print(f"Known-bad share: {bucket_failures.get('known-bad', 0) / bucket_failures.sum() * 100:.1f}% "
      "of failed-auth events come from synthetic Tor / scanner IP ranges.")


In [ ]:
# Cross-source pivot — pick a user, look at every class of activity
# they've touched. This is the "ASN/principal pivot" SOCs use to chase
# suspicious actors.
TARGET = "carol@corp.example.com"   # the brute-force scenario victim
pivot = DS.to_table(
    columns=["time", "class_uid", "class_name", "actor", "user", "src_endpoint", "status_id"]
).to_pandas()
pivot["actor"]    = pivot["actor"].apply(lambda d: (d or {}).get("user", {}).get("name") if isinstance(d, dict) else None)
pivot["user_n"]   = pivot["user"].apply(lambda d: (d or {}).get("name") if isinstance(d, dict) else None)
pivot["src_ip"]   = pivot["src_endpoint"].apply(lambda d: (d or {}).get("ip") if isinstance(d, dict) else None)

mine = pivot[(pivot["actor"] == TARGET) | (pivot["user_n"] == TARGET)].copy()
mine["dt"] = pd.to_datetime(mine["time"], unit="ms", utc=True)
print(f"Activity for {TARGET}:")
print(mine.groupby(["class_uid", "class_name"]).size().sort_values(ascending=False).head(10).to_string())
print()
print(f"Distinct source IPs they used: {mine['src_ip'].nunique()}")
print(f"Activity window:               {mine['dt'].min()}  →  {mine['dt'].max()}")


In [ ]:
# Anomaly: rolling failed-auth count detects the brute-force burst.
# Bucket failures into 5-minute windows; flag windows where ≥30 failures
# share a target user.
def find_brute_force_bursts(failed_df, window_min=5, threshold=30):
    f = failed_df[["dt", "actor", "src_ip"]].dropna(subset=["actor"]).copy()
    f["bucket"] = f["dt"].dt.floor(f"{window_min}min")
    by_window = f.groupby(["bucket", "actor"]).size().reset_index(name="count")
    return by_window[by_window["count"] >= threshold].sort_values("count", ascending=False)

bursts = find_brute_force_bursts(failed, window_min=5, threshold=30)
print(f"Brute-force bursts detected (≥30 failures in 5min, same user): {len(bursts)}")
print()
print(bursts.head(10).to_string(index=False))


## §3 — Detection-rule prototyping

Write Sigma-style rules in pandas and evaluate them against the
synthetic dataset. Each rule's output is the alert set it would
generate; you can eyeball precision/recall against the known injected
anomalies.

In [ ]:
# Rule 1 — Brute force authentication
# Definition: ≥20 failed auth events targeting the same user, from the
# same source IP, within a 1-hour window.
def rule_brute_force(failed_df, window_h=1, threshold=20):
    f = failed_df[["dt", "actor", "src_ip"]].dropna().copy()
    f["bucket"] = f["dt"].dt.floor(f"{window_h}h")
    g = f.groupby(["bucket", "actor", "src_ip"]).size().reset_index(name="failures")
    return g[g["failures"] >= threshold].sort_values("failures", ascending=False)

alerts_brute = rule_brute_force(failed, window_h=1, threshold=20)
print(f"Rule 1 — Brute force: {len(alerts_brute)} alerts")
print(alerts_brute.head(10).to_string(index=False))


In [ ]:
# Rule 2 — Off-hours sensitive API activity (CloudTrail / Vault api_activity).
# Definition: sensitive operations (Get*Secret, AssumeRole, *Policy*, *Key*)
# between 22:00 and 06:00 UTC.
api = DS.to_table(
    filter=(ds.field("class_uid") == 6003),  # API Activity
    columns=["time", "actor", "api"],
).to_pandas()
api["dt"]        = pd.to_datetime(api["time"], unit="ms", utc=True)
api["hour"]      = api["dt"].dt.hour
api["actor"]     = api["actor"].apply(lambda d: (d or {}).get("user", {}).get("name") if isinstance(d, dict) else None)
api["operation"] = api["api"].apply(lambda d: (d or {}).get("operation") if isinstance(d, dict) else None)

SENSITIVE = ("GetSecretValue","AssumeRole","AttachRolePolicy","DeleteAccessKey",
             "CreateAccessKey","PutObjectAcl","DescribeInstances","ListBuckets")
sensitive_off_hours = api[
    (api["operation"].isin(SENSITIVE)) &
    ((api["hour"] >= 22) | (api["hour"] < 6))
]
print(f"Rule 2 — Off-hours sensitive API: {len(sensitive_off_hours)} alerts")
print()
print("Top actors:")
print(sensitive_off_hours.groupby(["actor","operation"]).size().sort_values(ascending=False).head(10).to_string())


In [ ]:
# Rule 3 — Vault permission-denied spike
# Definition: ≥5 denied responses from the same entity_id in a 10-minute
# window. Catches the injected rogue-svc-account scenario.
vault = DS.to_table(
    filter=(ds.field("class_uid") == 6003),
    columns=["time", "actor", "metadata", "status_detail"],
).to_pandas()
vault["dt"]      = pd.to_datetime(vault["time"], unit="ms", utc=True)
vault["actor"]   = vault["actor"].apply(lambda d: (d or {}).get("user", {}).get("name") if isinstance(d, dict) else None)
vault["product"] = vault["metadata"].apply(lambda d: ((d or {}).get("product") or {}).get("name") if isinstance(d, dict) else None)

vault_denied = vault[
    (vault["product"] == "HashiCorp Vault") &
    (vault["status_detail"].fillna("").str.contains("permission denied", case=False, na=False))
].copy()
vault_denied["bucket"] = vault_denied["dt"].dt.floor("10min")

vault_alerts = (vault_denied.groupby(["bucket", "actor"]).size()
                  .reset_index(name="denials")
                  .query("denials >= 5")
                  .sort_values("denials", ascending=False))
print(f"Rule 3 — Vault permission-denied burst: {len(vault_alerts)} alerts")
print(vault_alerts.head(10).to_string(index=False))


In [ ]:
# Rule 4 — Multi-region API activity (lateral movement signal)
# Definition: same principal making API calls across ≥3 distinct cloud
# regions within a 10-minute window. Catches the injected lateral-movement.
api_geo = DS.to_table(
    filter=(ds.field("class_uid") == 6003),
    columns=["time", "actor", "cloud"],
).to_pandas()
api_geo["dt"]     = pd.to_datetime(api_geo["time"], unit="ms", utc=True)
api_geo["actor"]  = api_geo["actor"].apply(lambda d: (d or {}).get("user", {}).get("name") if isinstance(d, dict) else None)
api_geo["region"] = api_geo["cloud"].apply(lambda d: (d or {}).get("region") if isinstance(d, dict) else None)
api_geo["bucket"] = api_geo["dt"].dt.floor("10min")

mreg = (api_geo.dropna(subset=["actor","region"])
        .groupby(["bucket","actor"])["region"].nunique()
        .reset_index(name="distinct_regions"))
multi_region = mreg[mreg["distinct_regions"] >= 3].sort_values("distinct_regions", ascending=False)
print(f"Rule 4 — Multi-region API activity: {len(multi_region)} alerts")
print(multi_region.head(10).to_string(index=False))


## §4 — Compliance evidence

Three things compliance audits typically ask for:
1. **Severity distribution** — proves the SIEM is seeing a realistic
   mix of low/medium/high/critical, not all noise or all alerts.
2. **MITRE ATT&CK coverage** — which techniques does the ingest
   surface? Gaps = blind spots.
3. **Audit completeness** — every event has the fields required for
   forensic reconstruction (`time`, `metadata`, identity, source).


In [ ]:
# Severity distribution across all events.
sev = DS.to_table(columns=["severity_id", "severity", "category_name"]).to_pandas()

# Drop the absurd severities (legacy / odd mappings) and sort canonically.
SEV_ORDER = ["Informational","Low","Medium","High","Critical","Other"]
sev_clean = sev[sev["severity"].isin(SEV_ORDER)]

by_sev = sev_clean["severity"].value_counts().reindex(SEV_ORDER, fill_value=0)
print(by_sev.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = {"Informational":"#57606a","Low":"#1f883d","Medium":"#bf8700",
          "High":"#cf222e","Critical":"#a4072e","Other":"#888"}
by_sev.plot(kind="bar", ax=ax, color=[colors.get(s,"#888") for s in by_sev.index])
ax.set_ylabel("events")
ax.set_title("Severity distribution — all classes")
ax.set_yscale("log")
plt.setp(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# MITRE ATT&CK technique coverage from finding-class events.
# CrowdStrike Falcon mapping puts MITRE techniques into the raw_data;
# CEF/LEEF/Suricata findings carry them in finding_info.types or
# raw_data. We look at the raw_data text as a fallback.
findings = DS.to_table(
    filter=(ds.field("class_uid") == 2004),  # Detection Finding
    columns=["time", "finding_info", "raw_data", "metadata"],
).to_pandas()

import re
TECH_RX = re.compile(r"T\d{4}(?:\.\d{3})?")
def find_techniques(row):
    blobs = []
    fi = row.get("finding_info")
    if isinstance(fi, dict):
        blobs.append(str(fi))
    rd = row.get("raw_data")
    if rd is not None:
        blobs.append(str(rd))
    techs = set()
    for b in blobs:
        techs.update(TECH_RX.findall(b))
    return list(techs)

findings["techniques"] = findings.apply(find_techniques, axis=1)
all_techs = [t for techs in findings["techniques"] for t in techs]
tech_freq = pd.Series(all_techs).value_counts()

print(f"Findings analysed:     {len(findings):,}")
print(f"Distinct techniques:   {tech_freq.size}")
print()
print("Top 15 MITRE ATT&CK techniques observed:")
print(tech_freq.head(15).to_string())


In [ ]:
# Audit completeness — what fraction of events have the forensic basics?
# These are the fields an investigator needs at minimum:
#   - time (epoch ms)
#   - metadata.product.name
#   - metadata.uid (deduplication primary key)
#   - actor.user.name OR user.name (who did it?)
#   - src_endpoint.ip OR cloud.account.uid (where from?)
forensic = DS.to_table(
    columns=["time","metadata","actor","user","src_endpoint","cloud","raw_data"],
).to_pandas()

def has_id(d, *path):
    cur = d
    for p in path:
        if not isinstance(cur, dict): return False
        cur = cur.get(p)
    return cur is not None

forensic["has_time"]    = forensic["time"].notna()
forensic["has_uid"]     = forensic["metadata"].apply(lambda d: has_id(d, "uid"))
forensic["has_product"] = forensic["metadata"].apply(lambda d: has_id(d, "product", "name"))
forensic["has_actor"]   = (
    forensic["actor"].apply(lambda d: has_id(d, "user", "name")) |
    forensic["user"].apply(lambda d: has_id(d, "name"))
)
forensic["has_source"]  = (
    forensic["src_endpoint"].apply(lambda d: has_id(d, "ip")) |
    forensic["cloud"].apply(lambda d: has_id(d, "account", "uid"))
)
forensic["has_raw"]     = forensic["raw_data"].notna()

print("Forensic field completeness across the dataset:")
for c in ["has_time","has_uid","has_product","has_actor","has_source","has_raw"]:
    pct = forensic[c].mean() * 100
    print(f"  {c:<14} {pct:5.1f}%")
print()
print(f"Total events analysed: {len(forensic):,}")
print()
print("Interpretation: anything below ~90% on has_actor or has_source means")
print("that class of mapping needs work — the linter already gates on metadata + time.")


In [ ]:
# Summary — overall takeaways from the four sections.
print(f"Dataset:                      {total_rows:,} events / {total_bytes/1024/1024/1024:.2f} GB Parquet")
print(f"OCSF classes covered:         {len(class_uids)}")
print(f"Time window:                  {len(days)} days  ({days[0]} → {days[-1]})")
print()
print("Detection-rule outputs:")
print(f"  Brute force            : {len(alerts_brute):>4} alerts")
print(f"  Off-hours sensitive API: {len(sensitive_off_hours):>4} alerts")
print(f"  Vault denied burst     : {len(vault_alerts):>4} alerts")
print(f"  Multi-region API       : {len(multi_region):>4} alerts")
print()
print("Compliance:")
print(f"  Severity distribution  : seen across {len(by_sev[by_sev>0])} levels")
print(f"  MITRE techniques       : {tech_freq.size if tech_freq.size else 0} distinct")
print(f"  Forensic completeness  : has_actor={forensic['has_actor'].mean()*100:.0f}%, "
      f"has_source={forensic['has_source'].mean()*100:.0f}%")
